# Evidence-bound MCP server release gate

A signature, SBOM, scanner report, or provenance document is a claim—not deployment authority. This notebook verifies that every authenticated claim names the same artifact and release context before one digest-addressed promotion.

## Learning objectives

Rehash artifact bytes, verify signer identity and issuer, validate in-toto/SLSA provenance, inspect CycloneDX inventory and graph completeness, bind vulnerability results and exceptions, consume promotion authority once, and measure completed unsafe outcomes.

In [ ]:
import json
import runpy
from datetime import timedelta
from pathlib import Path

lab = runpy.run_path(Path('lab.py'))
env = lab['build_demo_environment']()
print('Loaded deterministic Course 12 evidence and policy fixtures')

## 1. Begin with trusted release intent and actual bytes

The release orchestrator fixes artifact, source, materials, inventory, environment, and repository. Evidence cannot rewrite that intent.

In [ ]:
{
    'release_id': env.intent.release_id,
    'artifact_digest': env.intent.artifact.digest,
    'rehash': lab['digest_bytes'](env.artifact_bytes),
    'source_revision': env.intent.source_revision,
    'target': env.intent.target_repository,
    'expected_components': sorted(env.intent.expected_components),
}

## 2. Verify cryptography and identity policy

The lab uses real Ed25519 verification. The allowed signer identity, OIDC issuer, public key, and revocation state come from policy-owned trust configuration.

In [ ]:
authorization = env.gate.verify_and_authorize(
    env.intent, env.artifact_bytes, env.evidence, env.now
)
{
    'signer': env.evidence.artifact_signature.signer_identity,
    'issuer': env.evidence.artifact_signature.oidc_issuer,
    'authorization_digest': authorization.evidence_digest,
    'expires_at': authorization.expires_at.isoformat(),
}

In [ ]:
tampered = lab['build_demo_environment']()
try:
    tampered.gate.verify_and_authorize(
        tampered.intent, tampered.artifact_bytes + b'malware', tampered.evidence, tampered.now
    )
    tamper_result = 'unexpected allow'
except lab['SupplyChainDenied'] as exc:
    tamper_result = exc.code
tamper_result

## 3. Check provenance semantics, not only its envelope

The in-toto subject, builder, build type, source URI/revision, lockfile, base image, and timestamps must all match trusted expectations.

In [ ]:
provenance = env.evidence.provenance.document
{
    'statement_type': provenance.statement_type,
    'predicate_type': provenance.predicate_type,
    'subject': provenance.subject[0].model_dump(),
    'builder': provenance.predicate.run_details.builder.id,
    'materials': [item.model_dump() for item in provenance.predicate.build_definition.resolved_dependencies],
}

In [ ]:
builder_attack = lab['build_demo_environment'](builder='https://attacker.example/builder')
try:
    builder_attack.gate.verify_and_authorize(
        builder_attack.intent, builder_attack.artifact_bytes, builder_attack.evidence, builder_attack.now
    )
    builder_result = 'unexpected allow'
except lab['SupplyChainDenied'] as exc:
    builder_result = exc.code
builder_result

## 4. Treat the SBOM as a bound dependency graph

A file named `bom.json` is not enough. The gate checks the root digest, complete composition, unique PURLs/references, graph reachability, exact locked inventory, and licenses.

In [ ]:
sbom = env.evidence.sbom.document
{
    'format': f'{sbom.bom_format} {sbom.spec_version}',
    'root': sbom.metadata.component.purl,
    'root_hash': sbom.metadata.component.hashes[0].content,
    'components': [component.purl for component in sbom.components],
    'complete': sbom.compositions[0].aggregate,
    'sbom_digest': lab['document_digest'](sbom),
}

In [ ]:
omission = lab['build_demo_environment']()
sbom_data = omission.evidence.sbom.document.model_dump(mode='json', by_alias=True)
sbom_data['dependencies'][0]['dependsOn'].pop()
changed_sbom = lab['CycloneDxBom'].model_validate(sbom_data)
changed_evidence = lab['replace_signed_document'](omission, 'sbom', changed_sbom)
try:
    omission.gate.verify_and_authorize(
        omission.intent, omission.artifact_bytes, changed_evidence, omission.now
    )
    omission_result = 'unexpected allow'
except lab['SupplyChainDenied'] as exc:
    omission_result = exc.code
omission_result

## 5. Bind scanner state, findings, VEX-like status, and exceptions

Scanner name/version, scan time, advisory-database age, artifact digest, and SBOM digest are policy inputs. Exceptions live in a trusted registry; they are not flags inside scanner output.

In [ ]:
critical = lab['VulnerabilityFinding'](
    vulnerability_id='GHSA-demo-critical', purl='pkg:pypi/httpx@0.28.1',
    severity='CRITICAL', fixed_version='0.28.2', exploitability='affected',
)
vulnerable = lab['build_demo_environment'](findings=(critical,))
try:
    vulnerable.gate.verify_and_authorize(
        vulnerable.intent, vulnerable.artifact_bytes, vulnerable.evidence, vulnerable.now
    )
    critical_result = 'unexpected allow'
except lab['SupplyChainDenied'] as exc:
    critical_result = exc.code
critical_result

In [ ]:
high = critical.model_copy(update={'vulnerability_id': 'GHSA-demo-high', 'severity': 'HIGH'})
exception_env = lab['build_demo_environment'](findings=(high,))
exception_env.exceptions.add(lab['ExceptionRecord'](
    exception_id='exc-1042', artifact_digest=exception_env.intent.artifact.digest,
    vulnerability_id=high.vulnerability_id, purl=high.purl,
    environment='production', owner='runtime-platform', ticket='RISK-1042',
    rationale='No compatible fixed version exists during the emergency release window.',
    compensating_controls=('disable outbound callbacks', 'monitor affected route'),
    approved_by='security-approver@example.com', policy_version=exception_env.policy.version,
    issued_at=exception_env.now - timedelta(hours=1),
    expires_at=exception_env.now + timedelta(hours=2), state='approved',
))
exception_authorization = exception_env.gate.verify_and_authorize(
    exception_env.intent, exception_env.artifact_bytes, exception_env.evidence, exception_env.now
)
exception_authorization.artifact_digest

## 6. Promote one exact digest once

The promotion service rehashes bytes, checks policy and target binding, looks up server-side authorization state, and atomically consumes the grant.

In [ ]:
immutable_ref = env.promotion.promote(
    authorization, env.artifact_bytes, env.intent.target_repository, 'v1.4.2',
    env.now + timedelta(minutes=1),
)
{
    'deployment_reference': immutable_ref,
    'stored_digest': env.promotion.tags[(env.intent.target_repository, 'v1.4.2')],
    'blob_matches': env.promotion.blobs[(env.intent.target_repository, env.intent.artifact.digest)] == env.artifact_bytes,
}

In [ ]:
try:
    env.promotion.promote(
        authorization, env.artifact_bytes, env.intent.target_repository, 'latest',
        env.now + timedelta(minutes=2),
    )
    replay_result = 'unexpected allow'
except lab['SupplyChainDenied'] as exc:
    replay_result = exc.code
replay_result

## 7. Evaluate completed outcomes with explicit denominators

Detector counts and denied logs are component metrics. A forbidden release completes only when unsafe bytes appear in registry/deployment state.

In [ ]:
cases = [
    ('safe', False, lambda e: (e.artifact_bytes, e.evidence)),
    ('tampered-bytes', True, lambda e: (e.artifact_bytes + b'evil', e.evidence)),
]
results = []
for name, unsafe, arrange in cases:
    case_env = lab['build_demo_environment']()
    payload, evidence = arrange(case_env)
    promoted = False
    try:
        grant = case_env.gate.verify_and_authorize(
            case_env.intent, payload, evidence, case_env.now
        )
        case_env.promotion.promote(
            grant, payload, case_env.intent.target_repository, name, case_env.now
        )
        promoted = True
    except lab['SupplyChainDenied']:
        pass
    results.append({'case': name, 'unsafe': unsafe, 'promoted': promoted})
unsafe_attempts = sum(item['unsafe'] for item in results)
unsafe_promotions = sum(item['unsafe'] and item['promoted'] for item in results)
safe_attempts = sum(not item['unsafe'] for item in results)
safe_blocks = sum(not item['unsafe'] and not item['promoted'] for item in results)
{
    'cases': results,
    'unsafe_release_rate': unsafe_promotions / unsafe_attempts,
    'safe_block_rate': safe_blocks / safe_attempts,
    'populations': {'unsafe_attempts': unsafe_attempts, 'safe_attempts': safe_attempts},
}

In [ ]:
[event.model_dump(mode='json') for event in [*env.gate.audit, *env.promotion.audit]]

## Production mapping

Replace local signing with Cosign/Sigstore or your managed PKI; validate in-toto/SLSA with maintained verifier libraries; generate SBOMs with Syft and validate CycloneDX/SPDX schemas; scan with OSV-Scanner, Grype, Trivy, or approved equivalents; evaluate policy with OPA/Cedar/Kyverno/Gatekeeper; attach evidence to OCI digests; and enforce again at deployment admission. Preserve the same subject, identity, freshness, exception, and one-use promotion invariants.

## Reflection and exercises

1. Why can a valid signature and a critical vulnerability both be true?
2. Which evidence edge prevents scanning one SBOM and deploying another artifact?
3. Add an authenticated OpenVEX document with issuer, product, vulnerability, status, justification, evidence, and expiry.
4. Compare a lockfile SBOM with a built-image SBOM and define a meaningful completeness denominator.
5. Add recovery for an unknown registry outcome without replaying the external effect.